# Lesson 5: attention closeup

Stage 5 - attention, computed by hand for one head, then drawn as a heat map.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Normansrule/transparent-transformer-llm/blob/main/notebooks/05_attention_closeup.ipynb) &nbsp; [lesson page](../stages/05_attention_closeup/) &nbsp;|&nbsp; [live in your browser](https://Normansrule.github.io/transparent-transformer-llm/#5)

The output below was produced by the model saved in this repository. Run the cells to reproduce it, then change things.

In [1]:
# Setup: works on GitHub Codespaces, on your own machine, and on Google Colab (where it clones the repository first).
import os, sys
if not os.path.exists("transparent_transformer"):
    if os.path.exists("../transparent_transformer"):
        os.chdir("..")
    else:
        os.system("git clone -q https://github.com/Normansrule/transparent-transformer-llm.git")
        os.chdir("transparent-transformer-llm")
sys.path.insert(0, os.getcwd())
print("ready, working in", os.getcwd())

ready, working in /home/claude/transparent-transformer-llm


In [2]:
import numpy as np

from transparent_transformer import GPT, BPETokenizer, paths
from transparent_transformer.alignment import format_prompt

tok = BPETokenizer.load(paths.TOKENIZER)
model = GPT.load(paths.ALIGNED_MODEL)
ids = tok.encode(format_prompt("What is the weather in Los Angeles?"))
pieces = [tok.token_str(i) for i in ids]
model.forward(np.array([ids]), capture=True)

# ---- redo block 1, head 1 by hand, in six lines --------------------------------------------
blk = model.blocks[0]
x = blk.ln1.forward(model.captured["stream"][0])[0]            # (T, d)  normalised token vectors
hd = model.cfg.d_head
qkv = x @ blk.attn.qkv.params["W"] + blk.attn.qkv.params["b"]   # (T, 3d)
d = model.cfg.d_model
Q, K = qkv[:, 0:hd], qkv[:, d:d + hd]                           # head 1 = the first d_head columns of Q and of K
scores = Q @ K.T / np.sqrt(hd)                                  # how well does each key answer each query?
scores[np.triu(np.ones_like(scores, dtype=bool), k=1)] = -1e9   # causal mask: hide the future
weights = np.exp(scores - scores.max(-1, keepdims=True))
weights /= weights.sum(-1, keepdims=True)                       # softmax: each row sums to 1
assert np.allclose(weights, model.captured["attention"][0][0, 0], atol=1e-5)
print("recomputed block 1 / head 1 by hand. It matches the model exactly.\n")

shades = " ░▒▓█"
print("rows = the token doing the looking, columns = the token being looked at")
print("the empty upper-right triangle is the causal mask: no token can see the future\n")
for i, p in enumerate(pieces):
    row = "".join(shades[min(4, int(w * 4.999))] * 2 for w in weights[i, :i + 1])
    print(f"   {p.strip()[:12]:>13} |{row}")

print("\nwhat the final token reads from, per head (it is about to write the answer):")
for li, att in enumerate(model.captured["attention"]):
    for h in range(att.shape[1]):
        w = att[0, h, -1]
        top = np.argsort(-w)[:3]
        print(f"   block {li+1} head {h+1}: " + "   ".join(f"{pieces[j]!r} {w[j]:.0%}" for j in top))

recomputed block 1 / head 1 by hand. It matches the model exactly.

rows = the token doing the looking, columns = the token being looked at
the empty upper-right triangle is the causal mask: no token can see the future

        <|user|> |██
               W |░░▓▓
             hat |▒▒░░  
              is |    ██  
             the |░░░░░░    
         weather |▒▒  ▒▒      
              in |░░  ▓▓        
             Los |  ░░            
         Angeles |        ▒▒  ░░    
               ? |                    
    <|assistant| |          ██          

what the final token reads from, per head (it is about to write the answer):
   block 1 head 1: ' weather' 88%   ' the' 6%   ' in' 3%
   block 1 head 2: '<|assistant|>' 16%   '<|user|>' 12%   'W' 12%
   block 1 head 3: ' Angeles' 46%   'W' 27%   'hat' 8%
   block 1 head 4: '?' 27%   ' is' 25%   ' the' 17%
   block 2 head 1: ' the' 16%   'hat' 13%   '?' 13%
   block 2 head 2: ' weather' 19%   ' the' 17%   ' is' 15%
   block 2 head 3: '

## Your turn

**Think first:** Why can a token only attend to tokens before it?

Then open `classroom/exercises/ex05.py`, fill in the function, and run the cell below to grade it.

In [3]:
!python classroom/check.py | head -14

## Homework: 0 of 10 passed

| exercise | lesson | result |
|---|---|---|
| `ex01.py` | Input | ⬜ not started |
| `ex02.py` | Tokenization | ⬜ not started |
| `ex03.py` | Embedding | ⬜ not started |
| `ex04.py` | Transformer | ⬜ not started |
| `ex05.py` | Attention | ⬜ not started |
| `ex06.py` | Pretraining | ⬜ not started |
| `ex07.py` | Backpropagation | ⬜ not started |
| `ex08.py` | Alignment | ⬜ not started |
| `ex09.py` | Sampling | ⬜ not started |
| `ex10.py` | Output | ⬜ not started |
Traceback (most recent call last):
  File "/home/claude/transparent-transformer-llm/classroom/check.py", line 76, in <module>
    print(summary)
BrokenPipeError: [Errno 32] Broken pipe
